# Machine Translation: English to Kamba (and Kamba to English)

This notebook trains an NMT model using `google/mt5-small` to translate between English and Kamba using the Hugging Face `transformers` library.

In [1]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset('michsethowusu/english-kamba_sentence-pairs_mt560')

# Display the structure and a few examples
print(dataset)
if 'train' in dataset:
    print("\nSample:", dataset['train'][0])
else:
    print("\nSample:", dataset[0])

DatasetDict({
    train: Dataset({
        features: ['eng', 'kam'],
        num_rows: 51054
    })
})

Sample: {'eng': '" Happy are the mild - tempered , since they will inherit the earth . " - Matt .', 'kam': '" Nĩ aathime ala auu : nũndũ nĩo makatiĩwa nthĩ . " - Mt .'}


## 2. Preprocessing
We need to tokenize the inputs and targets using `AutoTokenizer`. For mT5, we prefix tasks to guide translation.

In [2]:
from transformers import AutoTokenizer
import multiprocessing
import warnings
warnings.filterwarnings("ignore", message=".*byte fallback option.*")

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, legacy=False)

max_input_length = 128
max_target_length = 128
prefix = "translate English to Kamba: "

def preprocess_function(examples):
    keys = list(examples.keys())
    # Handle different dataset structures
    if 'translation' in keys:
        inputs = [prefix + ex['en'] for ex in examples['translation']]
        targets = [ex['kam'] for ex in examples['translation']]
    else:
        # Guess column names based on available keys
        en_candidates = ['english', 'en', 'English', 'source']
        kam_candidates = ['kamba', 'kam', 'Kamba', 'target']
        
        en_col = next((c for c in en_candidates if c in keys), keys[0])
        kam_col = next((c for c in kam_candidates if c in keys), keys[1] if len(keys)>1 else keys[0])
        
        inputs = [prefix + str(ex) for ex in examples[en_col]]
        targets = [str(ex) for ex in examples[kam_col]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    
    # Setup the tokenizer for targets using the modern API
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)
        
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

num_cores = multiprocessing.cpu_count()
print(f"Using {num_cores} cores for tokenization...")

# Tokenize dataset using multiprocessing!
tokenized_datasets = dataset.map(preprocess_function, batched=True, num_proc=num_cores)

Using 16 cores for tokenization...


## 3. Model Setup & Training

In [ ]:
import torch
import multiprocessing
try:
    multiprocessing.set_start_method("fork", force=True)
except RuntimeError:
    pass
torch.set_num_threads(16)
torch.set_num_interop_threads(8)
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

import warnings
warnings.filterwarnings("ignore", message="Can\'t initialize NVML")

# Check if GPU is actually available
device = "cpu"
print(f"--- Initializing Model ---")
print(f"Hardware Device: {device.upper()}")

# Load model and suppress the word embeddings warning
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, tie_word_embeddings=False)
from peft import get_peft_model, LoraConfig, TaskType
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

batch_size = 16
args = Seq2SeqTrainingArguments(
    "mt5-english-kamba",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=False,
    use_cpu=True,
    bf16=True, 
    # Speed Optimization 2: Dataloader Multiprocessing
    dataloader_num_workers=8,
    dataloader_prefetch_factor=4,
    # Speed Optimization 3: Only pin memory if on GPU to avoid PyTorch warnings
    dataloader_pin_memory=False,
)

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"].select(range(min(2500, len(tokenized_datasets["train"])))),
    eval_dataset=tokenized_datasets["validation"].select(range(min(500, len(tokenized_datasets["validation"])))) if "validation" in tokenized_datasets else tokenized_datasets["train"].select(range(min(500, len(tokenized_datasets["train"])))),
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Start training
trainer.train()

--- Initializing Model ---
Hardware Device: CPU


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


In [ ]:
# Save the trained model and tokenizer
save_directory = "./mt5-english-kamba-final"
print(f"Saving model and tokenizer to {save_directory}...")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

# Load the model and tokenizer from the saved directory for inference
print("Loading the saved model and tokenizer...")
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small", tie_word_embeddings=False)
model = PeftModel.from_pretrained(base_model, save_directory)
tokenizer = AutoTokenizer.from_pretrained(save_directory)

# Move model back to device (CPU/GPU)
import torch
device = "cpu"
model.to(device)
print("Model successfully loaded and ready for inference!")

## 4. Inference
Here we define our translation functions.

In [ ]:
import torch

def translate_en_to_kam(text):
    model.eval()
    input_text = prefix + text
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def translate_kam_to_en(text):
    model.eval()
    input_text = "translate Kamba to English: " + text
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_sentence = "How are you doing today?"
print("--- English -> Kamba Test ---")
print(f"English: {test_sentence}")
print(f"Kamba: {translate_en_to_kam(test_sentence)}")

### Interactive Translation (English -> Kamba)
Run the cell below to type your own English sentences and translate them to Kamba.

In [ ]:
user_input = input("Enter an English sentence to translate to Kamba: ")

if user_input.strip():
    translation = translate_en_to_kam(user_input)
    print("\n--- Results ---")
    print(f"English: {user_input}")
    print(f"Kamba:   {translation}")
else:
    print("No text entered!")

### Interactive Translation (Kamba -> English)
Run the cell below to type your own Kamba sentences and translate them to English.

In [ ]:
user_input = input("Enter a Kamba sentence to translate to English: ")

if user_input.strip():
    translation = translate_kam_to_en(user_input)
    print("\n--- Results ---")
    print(f"Kamba:   {user_input}")
    print(f"English: {translation}")
else:
    print("No text entered!")